In [1]:
%load_ext autoreload
%autoreload 2

### **Paso 1: Generación de datos**

In [2]:
# Generación de los datos sintéticos
from data_generation.data_config import DATA_CONFIG
from data_generation.PanelCreditSimulator import PanelCreditSimulator
from playground.generate_simple_panel import generate_panel

In [3]:
# Paso 1: generar los datos (por ahora hacemos uno muy sintético)
panel = generate_panel()

# Caso real
# panel_simulator = PanelCreditSimulator(DATA_CONFIG)
# panel = panel_simulator.simulate_panel()

In [4]:
# Paso 1.1: análisis exploratorio de los datos
print(panel.head())
print()
print(panel.info())
print()
print(list(panel.columns))

   firm_id  t    y_1     y_2  treated  control  cohort  cohort_start
0        0  0  7.367  46.123     True    False       0           8.0
1        0  1  9.411  46.683     True    False       0           8.0
2        0  2  7.898  53.131     True    False       0           8.0
3        0  3  9.999  47.314     True    False       0           8.0
4        0  4  8.462  50.540     True    False       0           8.0

<class 'pandas.DataFrame'>
RangeIndex: 3086 entries, 0 to 3085
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   firm_id       3086 non-null   int64  
 1   t             3086 non-null   int64  
 2   y_1           3086 non-null   float64
 3   y_2           3086 non-null   float64
 4   treated       3086 non-null   bool   
 5   control       3086 non-null   bool   
 6   cohort        3086 non-null   int64  
 7   cohort_start  1898 non-null   float64
dtypes: bool(2), float64(3), int64(3)
memory usage: 150.8 KB

### **Paso 2: Split en train y test**

In [5]:
# Divisón de IDs en train y test
from splits.split_generator import SplitGenerator

In [6]:
# Paso 2: generar el split train/test. El split_generator ya está configurado
# para que al train vayan todos los tratados y un porcentaje de los nini, y al
# test vayan todos los controles y el resto de los nini
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=13)
train_ids, test_ids = split_generator.generate()

In [7]:
last = panel.sort_values('t').groupby('firm_id').last()
status = last[['treated', 'control', 'cohort']].reset_index()
status

,firm_id,treated,control,cohort
0,0,True,False,0
1,1,False,False,-1
2,2,True,False,1
3,3,True,False,1
4,4,False,False,-1
...,...,...,...,...
195,195,True,False,0
196,196,False,False,-1
197,197,False,False,-1
198,198,False,False,-1


In [8]:
# Paso 2.1: revisar que el split se hizo correctamente
split = split_generator.split

train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

for id in treated:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == True, f"Firm {id} is not treated"
    assert firm["control"].iloc[0] == False, f"Firm {id} is control"

for id in control:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["control"].iloc[0] == True, f"Firm {id} is not control"
    assert firm["treated"].iloc[0] == False, f"Firm {id} is not treated"

for id in nini_train + nini_test:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == False and firm["control"].iloc[0] == False, f"Firm {id} is not NiNi"

### **Paso 3: Conversión de datos a tensores de PyTorch y formamos `Datasets`**

In [9]:
from connectors.lstm import LSTMConnector

In [10]:
lstm_connector = LSTMConnector(
    panel=panel,
    split=split_generator.split,
    feature_cols=['y_1', 'y_2']
)

train_dataset, test_dataset = lstm_connector.convert(fit_scaler=False)

In [30]:
print(len(train_dataset))
print(len(test_dataset))

print(train_dataset[0])
print(test_dataset[0])

180
300
(tensor([[ 7.3670, 46.1230],
        [ 9.4110, 46.6830],
        [ 7.8980, 53.1310],
        [ 9.9990, 47.3140],
        [ 8.4620, 50.5400],
        [ 9.6110, 48.0840],
        [ 8.7240, 51.8120],
        [ 9.1640, 49.7050]]), tensor(0), tensor(1.))
(tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]]), tensor(0), tensor(1.))


In [29]:
for (*X, y) in train_dataset[:10]:
    temporal = X[0]
    print(temporal.shape)   # Vemos que son secuencias de largo variable

# for (*X, y) in test_dataset[:10]:
#     temporal = X[0]
#     print(temporal.shape)

torch.Size([8, 2])
torch.Size([3, 2])
torch.Size([5, 2])
torch.Size([10, 2])
torch.Size([7, 2])
torch.Size([3, 2])
torch.Size([4, 2])
torch.Size([2, 2])
torch.Size([4, 2])
torch.Size([4, 2])


### **Paso 4: Creamos los DataLoaders**

Hacer esto no es tan directo porque tenemos secuencias de largo variable.
Existen dos alternativas:
1. Usar `padding` y `packing`.
2. Organizar los lotes de tal manera que cada lote tenga secuencias de la misma
longitud.

Por ahora, vamos con la opción 1.

Algunas referencias:
- [Discuss PyTorch - Different length sequences as batch input to LSTM](https://discuss.pytorch.org/t/different-length-sequences-as-batch-input-to-lstm/66119)
- [Discuss PyTorch - Understanding pack_padded_sequence and pad_packed_sequence](https://discuss.pytorch.org/t/understanding-pack-padded-sequence-and-pad-packed-sequence/4099/15)

In [43]:
# Lo probamos primero con un ejemplo
# Supongamos que nuestro lote es el siguiente
batch = [train_dataset[i][0] for i in range(3)]
for elem in batch:
    print(elem.shape)  # Vemos que son secuencias de largo variable

torch.Size([8, 2])
torch.Size([3, 2])
torch.Size([5, 2])


In [44]:
from torch.nn.utils.rnn import pad_sequence
batch_padded = pad_sequence(batch, batch_first=True)
for elem in batch_padded:
    print(elem.shape)  # Vemos que ahora todas las secuencias tienen el mismo largo

torch.Size([8, 2])
torch.Size([8, 2])
torch.Size([8, 2])


In [56]:
import torch

def collate_fn(batch):
    """
    batch: lista de (sequence, cohort, label)
    """
    sequences, cohorts, labels = zip(*batch)

    # Tenemos que devolver los largo originales para que el modelo sepa hasta
    # dónde leer (esto después se le pasa a pack_padded_sequence)
    lengths = torch.tensor([s.shape[0] for s in sequences], dtype=torch.long)

    # pad_sequence apila y rellena con 0s hasta la longitud máxima del batch
    # sequences_padded: (batch_size, T_max, F)
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)

    return (
        sequences_padded,
        lengths,
        torch.stack(cohorts),
        torch.stack(labels),
    )

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [68]:
# Veamos un batch de ejemplo
batch = next(iter(train_loader))

print(f"Secuencias: {batch[0].shape}\n")        # (batch_size, T_max, F)
print(f"Largos originales:\n\t{batch[1]}\n")    # (batch_size,)
print(f"IDs de cohortes: \n\t{batch[2]}\n")     # (batch_size,)
print(f"Labels: \n\t{batch[3]}")                # (batch_size,)

Secuencias: torch.Size([32, 9, 2])

Largos originales:
	tensor([6, 7, 7, 9, 1, 8, 4, 8, 3, 5, 6, 8, 2, 6, 3, 9, 1, 1, 8, 9, 4, 5, 4, 1,
        9, 3, 2, 1, 6, 1, 2, 7])

IDs de cohortes: 
	tensor([2, 0, 2, 2, 0, 0, 0, 0, 1, 0, 2, 1, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 1,
        1, 1, 1, 1, 1, 1, 0, 2])

Labels: 
	tensor([0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0.,
        1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


### **Paso 5: Instanciar el modelo**

In [72]:
from models.lstm_classifier import LSTMClassifier

In [82]:
model = LSTMClassifier(
    n_features=2,
    lstm_hidden_size=64,
    lstm_num_layers=1,
    n_cohorts=3,
    dropout=0.3
)

In [97]:
# Probamos que el modelo ande bien con una muestra del dataset
seq, cohort, label = train_dataset[0]

print(seq.shape)
print(cohort.shape)

lengths = torch.tensor([len(seq)], dtype=torch.long)
print(lengths, lengths.shape)

# El modelo espera (batch_size, seq_len, n_features) así que agregamos la
# dimensión de batch
logit = model(seq.unsqueeze(0), lengths, cohort.unsqueeze(0))
print(logit)         # tensor con un valor
print(logit.shape)   # torch.Size([1, 1])

torch.Size([8, 2])
torch.Size([])
tensor([8]) torch.Size([1])
tensor([[-0.2317]], grad_fn=<AddmmBackward0>)
torch.Size([1, 1])


In [99]:
# Probemos con un batch
batch = next(iter(train_loader))
seqs, lengths, cohorts, labels = batch

logit = model(seqs, lengths, cohorts)
print(logit)         # tensor con un valor por cada muestra del batch
print(logit.shape)   # torch.Size([batch_size, 1])

tensor([[-0.2635],
        [-0.3653],
        [-0.3008],
        [-0.0058],
        [-0.2916],
        [-0.3061],
        [-0.2876],
        [-0.3332],
        [-0.2117],
        [-0.2571],
        [-0.1402],
        [-0.3513],
        [-0.2670],
        [-0.2141],
        [-0.2726],
        [ 0.0076],
        [-0.1658],
        [-0.1089],
        [-0.1830],
        [-0.2120],
        [-0.2635],
        [-0.1061],
        [-0.1889],
        [-0.1962],
        [-0.1186],
        [-0.1580],
        [-0.4817],
        [-0.2023],
        [-0.0841],
        [-0.1380],
        [-0.1543],
        [-0.2957]], grad_fn=<AddmmBackward0>)
torch.Size([32, 1])
